In [2]:
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import torch
from torch.amp import autocast
import time
import matplotlib.pyplot as plt
from model import ContinuousMotionModel
from dataset.dataset import *
import utils.utils as utils
import soundfile as sf
from IPython.display import clear_output
import os

device = utils.get_device()

In [3]:
# model_path = "trained_models/20-09-tests/rich_feature_speak_fingers_0__mv5ce2aw_epoch_2001.pth"
model_path = "trained_models/pretrained_standard_diffusion.pth"

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path, device)
model = model.to(device)
model.eval() # Set the model to evaluation mode

c:\python\311\Lib\site-packages\torch\nn\init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


ContinuousMotionModel(
  (timestep_mlp): Sequential(
    (0): Linear(in_features=1, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
  )
  (timestep_stacking_mlp): Sequential(
    (0): Linear(in_features=1, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
  )
  (style_linear): Linear(in_features=18, out_features=64, bias=True)
  (seed_linear): Linear(in_features=0, out_features=192, bias=True)
  (audio_linear): Linear(in_features=39, out_features=64, bias=True)
  (noisy_gesture_linear): Linear(in_features=1557, out_features=256, bias=True)
  (pre_local_attention_linear): Linear(in_features=576, out_features=256, bias=True)
  (multi_head_local_attention): LocalMHA(
    (to_qkv): Linear(in_features=256, out_features=768, bias=False)
    (attn_fn): LocalAttention(
      (dropout): Dropout(p=0.0, inplace=False)
      (rel_pos): SinusoidalEmbeddings()
    )
    (to_out): Linea

In [4]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
print(animation_visualisation.init_visualization(display=False))

http://localhost:8001/utils/animation/visualisation/new/animation_viewer.html?wsport=8703


In [ ]:
# Outpaint Diffusion inference loop
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=80,
        seed_length=8,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        loading_encoded_data=False,
        include_world_pos_gesture_features=True,
        include_vel_acc_features=True,
        return_audio_frame_index=True,  # Set to True to return the audio frame index
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, seed_gesture, _, main_agent_id_one_hot, finger_availability, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]
        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # Extract the first element from the tensor

        # Decode the input using the autoencoder model
        # encoded_gesture_seed = model.pose_encoder.encode(gesture_sequence)

        iteration_counter = 0
        
        current_sequence = gesture_sequence

        frame_start_time = time.time()
        while True:
            # Generate pure noise as the initial input
            
            overlap_frames = model.diffusion.overlap_frames

            current_sequence[:, :overlap_frames] = current_sequence[:, -overlap_frames:]
            current_sequence[:, overlap_frames:] = 0.0

            actual_audio_features = full_audio_features[start_frame + iteration_counter * dataset.seq_length: start_frame + iteration_counter * dataset.seq_length + dataset.seq_length, :].unsqueeze(0)

            denoised_gesture_sequence = current_sequence.clone()

            for timestep in range(model.diffusion.number_of_timesteps-1, -1, -30):

                print(f"Timestep: {timestep}")

                # apply diffusion at the current timestep
                noisy_gesture_sequence = model.diffusion.forward(denoised_gesture_sequence, timestep)

                # Now we apply the model to denoise the gesture sequence
                timestep_tensor = torch.tensor([timestep], dtype=torch.int64, device=device)
                denoised_gesture_sequence = model.forward(
                    timestep=timestep_tensor,
                    one_hot_style=main_agent_id_one_hot,
                    audio_features=actual_audio_features,
                    noisy_gesture_sequence=noisy_gesture_sequence,
                    seed_gesture_sequence=seed_gesture,
                    finger_availability=finger_availability
                )

                # Replace the overlap frames with the original sequence
                denoised_gesture_sequence[:, :overlap_frames] = current_sequence[:, :overlap_frames]

                animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0)[10:,:].to(torch.float32),noisy_gesture_sequence.squeeze(0)[10:,:100].to(torch.float32)), dim=1), "full tensor")
                time.sleep(0.025)

            ########################################################################################################################################################################
            current_sequence = denoised_gesture_sequence
            # Decode the output using the autoencoder model
            clear_output(wait=True)
            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0)[10:,:].to(torch.float32),denoised_gesture_sequence.squeeze(0)[10:,:100].to(torch.float32)), dim=1), "full tensor")

            # Add a small amount of noise to denoised_gesture_sequence
            noise = torch.randn_like(denoised_gesture_sequence) * 0.0075
            denoised_gesture_sequence = denoised_gesture_sequence + noise

            # In case the model added extra features for richer embeddings, we only take the original features for decoding
            sequence_to_decode = denoised_gesture_sequence[:,model.diffusion.overlap_frames:, :model.original_pose_features_per_frame]
        
            if model.pose_encoder is not None:
                unencoded_denoised_gesture_sequence = model.pose_encoder.decode(sequence_to_decode)
            else:
                unencoded_denoised_gesture_sequence = sequence_to_decode

            denmormalized_unencoded_denoised_gesture_sequence = dataset.skeleton.denormalize_poses(unencoded_denoised_gesture_sequence).squeeze(0).squeeze(0)

            # animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),denoised_gesture_sequence.squeeze(0)[:,:100].to(torch.float32)), dim=1), "full tensor")

            for frame in denmormalized_unencoded_denoised_gesture_sequence:
                # Start time for the current frame
                # Send each frame to the animation visualisation
                animation_visualisation.send_pose(frame.cpu(), dataset.skeleton)
                frame_end_time = time.time()

                frame_time = frame_end_time - frame_start_time

                # Print the time taken for the current frame
                # print(f"Frame {iteration_counter} processed in {frame_end_time - frame_start_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

                # Sleep for the remaining time in the 30 FPS frame
                time_to_sleep = max(0, (1/30) - (frame_time) - 0.0005)  # 0.01 is a small buffer to account for processing time
                time.sleep(time_to_sleep)
                frame_start_time = time.time()

            iteration_counter += 1
            print(f"Iteration: {iteration_counter}")

Iteration: 2
Timestep: 999
Timestep: 969
Timestep: 939
Timestep: 909
Timestep: 879
Timestep: 849


KeyboardInterrupt: 